# Synthetic MRI Generation — HuashanMyo Dataset

Parallel to `synthetic_mri_example.ipynb` but using the **HuashanMyo** dataset
(`eval_notebooks/HuashanMyo/`).

| Folder | Suffix | Content |
|---|---|---|
| `Water/` | `_0001.nii.gz` | Water MRI stacks |
| `Fat/`   | `_0000.nii.gz` | Fat MRI stacks |
| `Label/` | `.nii.gz`      | Segmentation masks, labels 0–11 |

The truly synthetic cell extends `_LABEL_PRIORS` with entries for labels 9–11
that are present in HuashanMyo but not in the default myosegmenTUM priors.

## Environment setup

Same environment as `synthetic_mri_example.ipynb`:

```bash
conda env create -f ../environment_creation.yml
conda activate mri_creation
python -m ipykernel install --user --name mri_creation --display-name mri_creation
jupyter notebook synthetic_mri_huashanmyo.ipynb
```

In [ ]:
import sys, pathlib
import numpy as np
import matplotlib.pyplot as plt
import SimpleITK as sitk

sys.path.insert(0, str(pathlib.Path('..', 'src').resolve()))

from dissector.creation import generate_augmented, generate_synthetic, register_and_blend

In [ ]:
EVAL_DIR   = pathlib.Path('.')
DATA_DIR   = EVAL_DIR / 'HuashanMyo'
OUTPUT_DIR = EVAL_DIR / 'synthetic_output' / 'huashanmyo'

# First subject
water_path  = DATA_DIR / 'Water' / 'THIGH_001_0001.nii.gz'
fat_path    = DATA_DIR / 'Fat'   / 'THIGH_001_0000.nii.gz'
seg_path    = DATA_DIR / 'Label' / 'THIGH_001.nii.gz'

# Second subject — for register-and-blend
water_path2 = DATA_DIR / 'Water' / 'THIGH_002_0001.nii.gz'
fat_path2   = DATA_DIR / 'Fat'   / 'THIGH_002_0000.nii.gz'

print('Water  exists:', water_path.exists())
print('Fat    exists:', fat_path.exists())
print('Label  exists:', seg_path.exists())
print('Water2 exists:', water_path2.exists())

## 1 — Augmented images

TorchIO augmentation using the real water + fat scan pair: flips, affine,
elastic deformation, bias field, noise, and gamma applied identically to
both channels.

In [ ]:
aug_pairs = generate_augmented(
    water_path=water_path,
    fat_path=fat_path,
    n=2,
    output_dir=OUTPUT_DIR / 'augmented',
)

print(f'\nGenerated {len(aug_pairs)} augmented pair(s):')
for w, f in aug_pairs:
    print(f'  water: {w.name}')
    print(f'  fat  : {f.name}')

In [ ]:
# Visual check — original water vs two augmented water images
def mid_slice(path):
    arr = sitk.GetArrayFromImage(sitk.ReadImage(str(path))).astype(float)
    s = arr[arr.shape[0] // 2]
    return (s - s.min()) / (s.max() - s.min() + 1e-8)

titles = ['Original water', 'Augmented 0 water', 'Augmented 1 water']
images = [water_path, aug_pairs[0][0], aug_pairs[1][0]]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, title, path in zip(axes, titles, images):
    ax.imshow(mid_slice(path), cmap='gray', origin='lower')
    ax.set_title(title)
    ax.axis('off')
fig.suptitle('Middle slice — original vs augmented water (HuashanMyo THIGH_001)', fontsize=12)
plt.tight_layout()
plt.show()

## 2 — Register and blend two augmented images

In [ ]:
blended = register_and_blend(
    fixed_path=aug_pairs[0][0],
    moving_path=aug_pairs[1][0],
    output_path=OUTPUT_DIR / 'blended.nii.gz',
    alpha=0.5,
    transform='affine',
)

print('Registration and blending complete.')

In [ ]:
titles = ['Augmented 0', 'Augmented 1', 'Blended (50/50)']
images = [aug_pairs[0][0], aug_pairs[1][0], OUTPUT_DIR / 'blended.nii.gz']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, title, path in zip(axes, titles, images):
    ax.imshow(mid_slice(path), cmap='gray', origin='lower')
    ax.set_title(title)
    ax.axis('off')
fig.suptitle('Middle slice — two augmented images and their registered blend', fontsize=12)
plt.tight_layout()
plt.show()

## 3 — Truly synthetic images from segmentation mask

HuashanMyo has labels 0–11 (11 muscle classes).  The default `_LABEL_PRIORS`
only covers 0–8, so labels 9–11 are patched in below before calling
`generate_synthetic`.

In [ ]:
import dissector.creation as _creation

# Extend priors for HuashanMyo labels 9-11
# Format: (water_mu_lo, water_mu_hi, water_sig, ff_mu_lo, ff_mu_hi, ff_sig)
_creation._LABEL_PRIORS.update({
    9:  (0.40, 0.75, 0.05, 0.03, 0.20, 0.03),
    10: (0.40, 0.75, 0.05, 0.03, 0.20, 0.03),
    11: (0.40, 0.75, 0.05, 0.03, 0.20, 0.03),
})
print('Label priors now cover labels:', sorted(_creation._LABEL_PRIORS))

In [ ]:
synth_pairs = generate_synthetic(
    seg_path=seg_path,
    n=2,
    output_dir=OUTPUT_DIR / 'truly_synthetic',
    seed=42,
)

print(f'\nGenerated {len(synth_pairs)} truly synthetic pair(s):')
for w, f in synth_pairs:
    print(f'  {w.name}')

In [ ]:
titles = ['Original', 'Truly synth 0', 'Truly synth 1']
images = [img_path, synth_pairs[0][0], synth_pairs[1][0]]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, title, path in zip(axes, titles, images):
    ax.imshow(mid_slice(path), cmap='gray', origin='lower')
    ax.set_title(title)
    ax.axis('off')
fig.suptitle('Middle slice — original vs truly synthetic (from segmentation mask)', fontsize=12)
plt.tight_layout()
plt.show()